# News Daily Features

Objetivo: transformar o dataset de notícias em granularidade intraday (`1 linha = 1 notícia`) em um dataset diário (`1 linha = 1 dia`).

### Entrada

`data/processed/energy_br.csv`

### Saída

`data/features/energy_news_daily.csv`

### Features iniciais

- `news_count`
- `signal`
- `avg_score`
- `avg_relevance`
- `positive_count`
- `neutral_count`
- `negative_count`
- `positive_share`
- `neutral_share`
- `negative_share`
- `max_relevance`
- `score_std`
- `signal_ma_3`
- `signal_ma_5`
- `signal_ma_7`

O `signal` diário segue a lógica:

```text
Σ(score × relevance)
────────────────────
    Σ(relevance)
```


In [12]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NEWS_PATH = PROJECT_ROOT / "data" / "processed" / "energy_br.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "features" / "energy_news_daily.csv"

print("News input :", NEWS_PATH)
print("Output     :", OUTPUT_PATH)


News input : /Users/hudsonborgo/Desktop/Can-AI-listen-to-the-market-improved/data/processed/energy_br.csv
Output     : /Users/hudsonborgo/Desktop/Can-AI-listen-to-the-market-improved/data/features/energy_news_daily.csv


## 1. Carregar as notícias


In [13]:
news = pd.read_csv(NEWS_PATH)

news["published_at"] = pd.to_datetime(
    news["published_at"],
    utc=True,
    errors="coerce",
)

news["score"] = pd.to_numeric(
    news["score"],
    errors="coerce",
)

news["relevance"] = pd.to_numeric(
    news["relevance"],
    errors="coerce",
)

print(f"Rows: {len(news):,}")
print(f"Period: {news['published_at'].min()} -> {news['published_at'].max()}")

news.head()


Rows: 2,352
Period: 2026-01-05 11:46:41+00:00 -> 2026-08-24 19:58:39+00:00


,source,title,summary,url,published_at,fetched_at,category,sentiment,score,relevance,reason,classified_at
0,megawhat,Clearing pode destravar liquidez e reduzir ris...,Sinal de preços no mercado livre A criação de ...,https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 18:23:48+00:00,2026-08-20T19:26:41+00:00,energy_br,neutral,0.05,0.22,A criação de uma clearing pode melhorar liquid...,2026-08-27T11:30:20.813925+00:00
1,megawhat,Data centers de 2 GW podem dar ‘tranco’ no sis...,Diretor de Planejamento do Operador Nacional d...,https://megawhat.uol.com.br/podcasts/minutomeg...,2026-08-20 18:09:01+00:00,2026-08-20T19:26:41+00:00,energy_br,positive,0.31,0.53,A notícia aponta para aumento potencial de car...,2026-08-27T11:30:20.833898+00:00
2,megawhat,Petrobras amplia contratos de acesso de tercei...,Instalações de gás da Petrobras / crédito: Pet...,https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 16:01:30+00:00,2026-08-20T19:26:41+00:00,energy_br,neutral,0.05,0.18,O anúncio trata de acesso de terceiros ao proc...,2026-08-27T11:30:20.691948+00:00
3,megawhat,"Termelétrica GNA II, de 1,7 GW de capacidade, ...","Com entrada em operação da GNA II, complexo no...",https://megawhat.uol.com.br/economia-e-politic...,2026-08-20 15:47:39+00:00,2026-08-20T19:26:41+00:00,energy_br,positive,0.80,0.90,A parada da GNA II retira de forma relevante 1...,2026-08-27T11:30:20.839956+00:00
4,megawhat,Terminais de GNL já têm concorrência e não pre...,"Lino Cançado, CEO da Eneva, no MinutoMega Talk...",https://megawhat.uol.com.br/podcasts/minutomeg...,2026-08-20 14:59:10+00:00,2026-08-20T19:26:41+00:00,energy_br,neutral,0.05,0.18,A notícia é בעיקר sobre debate regulatório no ...,2026-08-27T11:30:20.792310+00:00


## 2. Filtrar notícias analisadas

Para gerar features de sentimento, consideramos somente notícias com `score` e `relevance` disponíveis.

In [ ]:
analyzed["date"] = analyzed["published_at"].dt.date

print(f"Analyzed news: {len(analyzed):,}")
print(f"Unique days  : {analyzed['date'].nunique():,}")

analyzed[
    [
        "date",
        "source",
        "title",
        "sentiment",
        "score",
        "relevance",
    ]
].head()


Analyzed news: 2,352
Unique days  : 161


,date,source,title,sentiment,score,relevance
0,2026-08-20,megawhat,Clearing pode destravar liquidez e reduzir ris...,neutral,0.05,0.22
1,2026-08-20,megawhat,Data centers de 2 GW podem dar ‘tranco’ no sis...,positive,0.31,0.53
2,2026-08-20,megawhat,Petrobras amplia contratos de acesso de tercei...,neutral,0.05,0.18
3,2026-08-20,megawhat,"Termelétrica GNA II, de 1,7 GW de capacidade, ...",positive,0.80,0.90
4,2026-08-20,megawhat,Terminais de GNL já têm concorrência e não pre...,neutral,0.05,0.18


## 3. Contribuição ponderada por notícia

Cada notícia contribui para o sinal diário por:

`weighted_score = score × relevance`

A relevância funciona como peso na agregação.

In [15]:
analyzed["weighted_score"] = (
    analyzed["score"]
    * analyzed["relevance"]
)

analyzed[
    [
        "date",
        "title",
        "score",
        "relevance",
        "weighted_score",
    ]
].head()


,date,title,score,relevance,weighted_score
0,2026-08-20,Clearing pode destravar liquidez e reduzir ris...,0.05,0.22,0.0110
1,2026-08-20,Data centers de 2 GW podem dar ‘tranco’ no sis...,0.31,0.53,0.1643
2,2026-08-20,Petrobras amplia contratos de acesso de tercei...,0.05,0.18,0.0090
3,2026-08-20,"Termelétrica GNA II, de 1,7 GW de capacidade, ...",0.80,0.90,0.7200
4,2026-08-20,Terminais de GNL já têm concorrência e não pre...,0.05,0.18,0.0090


## 4. Features diárias

Agregamos as notícias por `published_at`.

A granularidade final passa a ser diária


In [16]:
sentiment_daily = (
    analyzed
    .assign(
        sentiment=analyzed["sentiment"]
        .astype(str)
        .str.lower()
    )
    .groupby(
        ["date", "sentiment"]
    )
    .size()
    .unstack(fill_value=0)
)

for column in [
    "positive",
    "neutral",
    "negative",
]:
    if column not in sentiment_daily.columns:
        sentiment_daily[column] = 0

sentiment_daily = sentiment_daily[
    [
        "positive",
        "neutral",
        "negative",
    ]
].rename(
    columns={
        "positive": "positive_count",
        "neutral": "neutral_count",
        "negative": "negative_count",
    }
)

sentiment_daily.head()


sentiment,positive_count,neutral_count,negative_count
date,,,
2026-01-05,2,10,2
2026-01-06,0,11,2
2026-01-07,0,12,2
2026-01-08,0,11,1
2026-01-09,1,10,1


In [17]:
daily = (
    analyzed
    .groupby("date")
    .agg(
        news_count=("score", "count"),
        weighted_sum=("weighted_score", "sum"),
        total_relevance=("relevance", "sum"),
        avg_score=("score", "mean"),
        avg_relevance=("relevance", "mean"),
        max_relevance=("relevance", "max"),
        score_std=("score", "std"),
    )
    .join(sentiment_daily)
    .reset_index()
)

daily["signal"] = np.where(
    daily["total_relevance"] > 0,
    daily["weighted_sum"] / daily["total_relevance"],
    0.0,
)

daily["positive_share"] = (
    daily["positive_count"] / daily["news_count"]
)

daily["neutral_share"] = (
    daily["neutral_count"] / daily["news_count"]
)

daily["negative_share"] = (
    daily["negative_count"] / daily["news_count"]
)

daily["score_std"] = daily["score_std"].fillna(0.0)

daily.head()


,date,news_count,weighted_sum,total_relevance,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,signal,positive_share,neutral_share,negative_share
0,2026-01-05,14,0.3632,3.83,0.023571,0.273571,0.88,0.263544,2,10,2,0.094830,0.142857,0.714286,0.142857
1,2026-01-06,13,-0.2642,2.28,-0.023077,0.175385,0.78,0.117430,0,11,2,-0.115877,0.000000,0.846154,0.153846
2,2026-01-07,14,-0.2110,2.60,-0.007857,0.185714,0.68,0.139404,0,12,2,-0.081154,0.000000,0.857143,0.142857
3,2026-01-08,12,0.0578,1.64,0.030833,0.136667,0.26,0.041878,0,11,1,0.035244,0.000000,0.916667,0.083333
4,2026-01-09,12,-0.5302,2.54,-0.019167,0.211667,0.88,0.225931,1,10,1,-0.208740,0.083333,0.833333,0.083333


## 5. Médias móveis

As médias móveis ajudam a reduzir ruído diário e capturar persistência do sinal.

Começamos com janelas simples de 3, 5 e 7 dias de observação.


In [18]:
daily = daily.sort_values("date").reset_index(drop=True)

for window in [3, 5, 7]:
    daily[f"signal_ma_{window}"] = (
        daily["signal"]
        .rolling(
            window=window,
            min_periods=1,
        )
        .mean()
    )

daily[
    [
        "date",
        "signal",
        "signal_ma_3",
        "signal_ma_5",
        "signal_ma_7",
    ]
].tail(10)


,date,signal,signal_ma_3,signal_ma_5,signal_ma_7
151,2026-08-11,-0.076329,-0.322719,-0.267538,-0.219308
152,2026-08-12,0.045482,-0.034225,-0.227098,-0.182765
153,2026-08-13,0.177435,0.048863,-0.149048,-0.159253
154,2026-08-14,-0.135701,0.029072,-0.012188,-0.156251
155,2026-08-17,0.014041,0.018592,0.004986,-0.123843
156,2026-08-18,0.040921,-0.026913,0.028436,-0.000854
157,2026-08-19,0.044164,0.033042,0.028172,0.015716
158,2026-08-20,0.280023,0.121703,0.048690,0.066624
159,2026-08-21,0.338270,0.220819,0.143484,0.108450
160,2026-08-24,-0.333716,0.094859,0.073932,0.035429


## 6. Dataset final

In [19]:
feature_columns = [
    "date",
    "news_count",
    "signal",
    "avg_score",
    "avg_relevance",
    "max_relevance",
    "score_std",
    "positive_count",
    "neutral_count",
    "negative_count",
    "positive_share",
    "neutral_share",
    "negative_share",
    "signal_ma_3",
    "signal_ma_5",
    "signal_ma_7",
]

daily_features = daily[feature_columns].copy()

numeric_columns = daily_features.select_dtypes(
    include="number"
).columns

daily_features[numeric_columns] = (
    daily_features[numeric_columns]
    .round(4)
)

daily_features.tail(10)


,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
151,2026-08-11,20,-0.0763,0.0065,0.2070,0.82,0.1403,0,19,1,0.0000,0.9500,0.0500,-0.3227,-0.2675,-0.2193
152,2026-08-12,13,0.0455,0.0385,0.1515,0.24,0.0291,0,13,0,0.0000,1.0000,0.0000,-0.0342,-0.2271,-0.1828
153,2026-08-13,12,0.1774,0.0642,0.2242,0.74,0.2102,1,10,1,0.0833,0.8333,0.0833,0.0489,-0.1490,-0.1593
154,2026-08-14,19,-0.1357,-0.0058,0.2216,0.78,0.1907,1,16,2,0.0526,0.8421,0.1053,0.0291,-0.0122,-0.1563
155,2026-08-17,18,0.0140,0.0267,0.1622,0.46,0.1066,1,16,1,0.0556,0.8889,0.0556,0.0186,0.0050,-0.1238
156,2026-08-18,11,0.0409,0.0364,0.1382,0.22,0.0206,0,11,0,0.0000,1.0000,0.0000,-0.0269,0.0284,-0.0009
157,2026-08-19,13,0.0442,0.0385,0.2162,0.58,0.1182,1,11,1,0.0769,0.8462,0.0769,0.0330,0.0282,0.0157
158,2026-08-20,15,0.2800,0.1233,0.2900,0.90,0.2665,3,11,1,0.2000,0.7333,0.0667,0.1217,0.0487,0.0666
159,2026-08-21,13,0.3383,0.1154,0.2623,0.88,0.2982,2,10,1,0.1538,0.7692,0.0769,0.2208,0.1435,0.1085
160,2026-08-24,22,-0.3337,-0.0664,0.2532,0.96,0.2649,1,18,3,0.0455,0.8182,0.1364,0.0949,0.0739,0.0354


## 7. Validações rápidas

- uma linha por dia;
- `signal` entre -1 e +1;
- shares entre 0 e 1;
- contagem de sentimentos igual a `news_count`.


In [20]:
assert daily_features["date"].is_unique

assert daily_features["signal"].between(
    -1,
    1,
).all()

for column in [
    "positive_share",
    "neutral_share",
    "negative_share",
]:
    assert daily_features[column].between(
        0,
        1,
    ).all()

sentiment_total = (
    daily_features["positive_count"]
    + daily_features["neutral_count"]
    + daily_features["negative_count"]
)

assert (
    sentiment_total
    == daily_features["news_count"]
).all()

print("✓ Validations passed")


✓ Validations passed


## 8. Persistir features diárias

`02_market_news_eda.ipynb`


In [21]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

daily_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(
    f"✓ Saved {len(daily_features):,} daily rows "
    f"to {OUTPUT_PATH}"
)


✓ Saved 161 daily rows to /Users/hudsonborgo/Desktop/Can-AI-listen-to-the-market-improved/data/features/energy_news_daily.csv


## 9. Resumo

O dataset final contém uma linha por dia com:

- intensidade e direção agregada das notícias;
- relevância média;
- volume de notícias;
- distribuição de sentimento;
- dispersão dos scores;
- médias móveis do sinal.


In [22]:
daily_features.head()

,date,news_count,signal,avg_score,avg_relevance,max_relevance,score_std,positive_count,neutral_count,negative_count,positive_share,neutral_share,negative_share,signal_ma_3,signal_ma_5,signal_ma_7
0,2026-01-05,14,0.0948,0.0236,0.2736,0.88,0.2635,2,10,2,0.1429,0.7143,0.1429,0.0948,0.0948,0.0948
1,2026-01-06,13,-0.1159,-0.0231,0.1754,0.78,0.1174,0,11,2,0.0000,0.8462,0.1538,-0.0105,-0.0105,-0.0105
2,2026-01-07,14,-0.0812,-0.0079,0.1857,0.68,0.1394,0,12,2,0.0000,0.8571,0.1429,-0.0341,-0.0341,-0.0341
3,2026-01-08,12,0.0352,0.0308,0.1367,0.26,0.0419,0,11,1,0.0000,0.9167,0.0833,-0.0539,-0.0167,-0.0167
4,2026-01-09,12,-0.2087,-0.0192,0.2117,0.88,0.2259,1,10,1,0.0833,0.8333,0.0833,-0.0849,-0.0551,-0.0551
